## Data preparation

Plate reader data is loaded using pr.load_platereader_data, which parses the raw instrument output and merges the platemap to annotate each well with its construct identity.

In [1]:
from cdk.analysis.cytosol import platereader as pr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pr.plot_setup()

In [2]:
mango_data = pd.read_csv("20251030_MangoTO1_mScarlet_data.csv")
mango_data["Time"] = pd.to_timedelta(mango_data["Time"])
mango_data["Experiment"] = "MangoIV"

broccoli_data = pd.read_csv("20251027_Broccoli_mScarlet_data.csv")
broccoli_data["Time"] = pd.to_timedelta(broccoli_data["Time"])
broccoli_data["Experiment"] = "Broccoli"

combined = pd.concat([mango_data, broccoli_data], ignore_index=True)

## Summary figure

A three-panel summary figure comparing transcription reporter quality across both aptamer systems and presenting the key biological result.

- **Panel 1 — Transcription reporter quality**: SNR plotted against lag CV (%) for all transcription reporter constructs on a log SNR scale. Each point represents a construct, coloured by reporter type (TO1-Biotin or DFHBI-1T). Points in the lower right represent optimal reporters with high SNR and low timing variability.

- **Panel 2 — Transcription reporter precision**: Lag CV (%) for all transcription reporter constructs grouped by reporter type. The dashed line indicates the 5% CV threshold below which timing measurements are considered sufficiently precise. Constructs are ordered by aptamer variant.

- **Panel 3 — Translation initiation delay**: The difference between the mScarlet lag time and the DFHBI-1T lag time for each Broccoli construct, representing the delay between detectable transcription onset and detectable translation onset. Error bars represent propagated standard deviation. Broccoli constructs only are shown as the MangoIV TO1-Biotin lag times are not sufficiently reliable for this comparison.

In [3]:
construct_order_qc = [
    "F30MangoIV", "F30MangoIV-RiboJ",
    "F30-2xBroccoli", "F30-2xBroccoli-RiboJ",
    "tdBroccoli", "tdBroccoli-RiboJ"
]
qc_df["Construct"] = pd.Categorical(qc_df["Construct"], categories=construct_order_qc, ordered=True)
qc_df = qc_df.sort_values("Construct").reset_index(drop=True)

# Reorder delay_df to match Broccoli colour scheme
construct_order_delay = ["F30-2xBroccoli", "F30-2xBroccoli-RiboJ", "tdBroccoli", "tdBroccoli-RiboJ"]
delay_df["Construct"] = pd.Categorical(delay_df["Construct"], categories=construct_order_delay, ordered=True)
delay_df = delay_df.sort_values("Construct").reset_index(drop=True)

# Broccoli colour scheme matching other figures
broccoli_colours = {
    "F30-2xBroccoli": "tab:blue",
    "F30-2xBroccoli-RiboJ": "tab:orange",
    "tdBroccoli": "tab:green",
    "tdBroccoli-RiboJ": "tab:red",
}

# Three panel figure
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1 — SNR vs Lag CV scatter
palette = {"TO1-Biotin": "tab:orange", "DFHBI-1T": "tab:green"}
markers = {"TO1-Biotin": "o", "DFHBI-1T": "s"}
for reporter, group in qc_df.groupby("Reporter"):
    axes[0].scatter(group["SNR"], group["Lag CV (%)"],
                   color=palette[reporter], marker=markers[reporter],
                   s=100, label=reporter, zorder=3)
    for _, row in group.iterrows():
        axes[0].annotate(row["Construct"],
                        (row["SNR"], row["Lag CV (%)"]),
                        textcoords="offset points", xytext=(6, 4), fontsize=8)
axes[0].set_xscale("log")
axes[0].set_xlabel("Signal-to-background ratio (SNR, log scale)")
axes[0].set_ylabel("Lag CV (%)")
axes[0].set_title("Transcription reporter quality")
axes[0].legend(title=None, frameon=False)

# Panel 2 — Lag CV bar plot
colours_p2 = [palette["TO1-Biotin"] if r == "TO1-Biotin" else palette["DFHBI-1T"]
              for r in qc_df["Reporter"]]
axes[1].bar(qc_df["Construct"], qc_df["Lag CV (%)"], color=colours_p2)
axes[1].axhline(5, color="black", linestyle="--", linewidth=0.8)
axes[1].set_xlabel("Construct")
axes[1].set_ylabel("Lag CV (%)")
axes[1].set_title("Transcription reporter precision")
axes[1].tick_params(axis='x', rotation=45)
legend_elements_p2 = [
    Patch(facecolor="tab:orange", label="TO1-Biotin"),
    Patch(facecolor="tab:green", label="DFHBI-1T"),
    plt.Line2D([0], [0], color='black', linestyle='--', linewidth=0.8, label='5% threshold')
]
axes[1].legend(handles=legend_elements_p2, frameon=False)

# Panel 3 — Translation initiation delay
for i, row in delay_df.iterrows():
    axes[2].bar(i, row["Delay (minutes)"],
                color=broccoli_colours.get(row["Construct"], "grey"),
                yerr=row["Std"], capsize=4, edgecolor="white")
axes[2].set_xticks(range(len(delay_df)))
axes[2].set_xticklabels(delay_df["Construct"], rotation=45, ha="right")
axes[2].set_ylabel("Delay (minutes)")
axes[2].set_title("Translation initiation delay")

sns.despine()
plt.tight_layout()
plt.savefig("./figures/DNA_toolkit_aptamers_summary_plot.png", dpi=600)
plt.show()

NameError: name 'qc_df' is not defined

## Combined QC table

A unified QC table combining metrics from both the MangoIV and Broccoli experiments for all constructs and reporter combinations. Metrics are defined in the individual experiment notebooks. Note that tMangoIV TO1-Biotin lag times are not reported as no lag phase is detectable. † indicates a suspected construct label swap pending experimental confirmation.

In [ ]:
# MangoIV TO1-Biotin kinetics (F30 only - tMangoIV has no detectable lag)
mango_TO1_kinetics = (
    combined[
        (combined["Experiment"] == "MangoIV") &
        (combined["Name"].str.contains("F30MangoIV")) &
        (combined["Read"] == "TO1-Biotin")
    ]
    .groupby(["Name", "Read", "Well"])
    .apply(pr.kinetic_analysis_per_well)
)

# MangoIV mScarlet kinetics
mango_mScarlet_kinetics = pr.kinetic_analysis(
    combined[
        (combined["Experiment"] == "MangoIV") &
        (combined["Name"] != "Negative") &
        (combined["Read"] == "mScarlet")
    ],
    group_by=["Name", "Read", "Well"]
)

# Broccoli aptamer kinetics
broccoli_aptamer_kinetics = pr.kinetic_analysis(
    combined[
        (combined["Experiment"] == "Broccoli") &
        (combined["Name"] != "Negative") &
        (combined["Read"] == "GFP-Gext")
    ],
    group_by=["Name", "Read", "Well"]
)

# Broccoli mScarlet kinetics
broccoli_mScarlet_kinetics = pr.kinetic_analysis(
    combined[
        (combined["Experiment"] == "Broccoli") &
        (combined["Name"] != "Negative") &
        (combined["Read"] == "mScarlet")
    ],
    group_by=["Name", "Read", "Well"]
)

# Extract lag times
mango_TO1_lag = mango_TO1_kinetics["Lag"]["Time"].dt.total_seconds() / 60
mango_mScarlet_lag = mango_mScarlet_kinetics["Lag"]["Time"].dt.total_seconds() / 60
broccoli_aptamer_lag = broccoli_aptamer_kinetics["Lag"]["Time"].dt.total_seconds() / 60
broccoli_mScarlet_lag = broccoli_mScarlet_kinetics["Lag"]["Time"].dt.total_seconds() / 60

# Means and stds
mango_TO1_mean = mango_TO1_lag.groupby("Name").mean()
mango_TO1_std = mango_TO1_lag.groupby("Name").std()
mango_mScarlet_mean = mango_mScarlet_lag.groupby("Name").mean()
mango_mScarlet_std = mango_mScarlet_lag.groupby("Name").std()
broccoli_aptamer_mean = broccoli_aptamer_lag.groupby("Name").mean()
broccoli_aptamer_std = broccoli_aptamer_lag.groupby("Name").std()
broccoli_mScarlet_mean = broccoli_mScarlet_lag.groupby("Name").mean()
broccoli_mScarlet_std = broccoli_mScarlet_lag.groupby("Name").std()

# Signal stability
peak_signal = combined[combined["Name"] != "Negative"].groupby(["Experiment", "Name", "Read", "Well"])["Data"].max().groupby(["Experiment", "Name", "Read"]).mean()
end_signal = combined.groupby(["Experiment", "Name", "Read"]).apply(
    lambda g: g.loc[g["Time"].idxmax(), "Data"]
).groupby(["Experiment", "Name", "Read"]).mean()
stability = (end_signal / peak_signal).drop("Negative", level="Name")

# SNR
steady_state_combined = pr.find_steady_state(combined, group_by=["Experiment", "Name", "Read", "Well"])
ss_data = steady_state_combined["Data_steadystate"]
neg_ss = ss_data.xs("Negative", level="Name").groupby(["Experiment", "Read"]).mean()
snr = ss_data.groupby(["Experiment", "Name", "Read"]).mean() / neg_ss
snr = snr.drop("Negative", level="Name")

# Build combined QC table
rows = []

# MangoIV constructs
mango_constructs = {
    "F30MangoIV-mScarlet": ("MangoIV", "TO1-Biotin", "Yes"),
    "F30MangoIV-RiboJ-mScarlet": ("MangoIV", "TO1-Biotin", "Yes"),
    "tMangoIV-mScarlet": ("MangoIV", "TO1-Biotin", "No"),
    "tMangoIV-RiboJ-mScarlet": ("MangoIV", "TO1-Biotin", "No†"),
}

for construct, (experiment, aptamer_read, lag_detectable) in mango_constructs.items():
    clean_name = construct.replace("-mScarlet", "")
    has_lag = lag_detectable == "Yes"

    # Aptamer row
    rows.append({
        "Experiment": experiment,
        "Construct": clean_name,
        "Reporter": aptamer_read,
        "Lag detectable": lag_detectable,
        "SNR": round(snr.loc[(experiment, construct, aptamer_read)], 2),
        "R²": round(mango_TO1_kinetics.loc[(construct, aptamer_read), "Fit"]["R^2"].mean(), 4) if has_lag else float("nan"),
        "Lag mean (min)": round(mango_TO1_mean.get(construct, 0), 1) if has_lag else "N/A",
        "Lag std (min)": round(mango_TO1_std.get(construct, 0), 1) if has_lag else "N/A",
        "Lag CV (%)": round(mango_TO1_std.get(construct, 0) / mango_TO1_mean.get(construct, 1) * 100, 1) if has_lag else "N/A",
        "Signal stability": round(stability.loc[(experiment, construct, aptamer_read)], 2),
    })

    # mScarlet row
    rows.append({
        "Experiment": experiment,
        "Construct": clean_name,
        "Reporter": "mScarlet",
        "Lag detectable": "Yes",
        "SNR": round(snr.loc[(experiment, construct, "mScarlet")], 2),
        "R²": round(mango_mScarlet_kinetics.loc[(construct, "mScarlet"), "Fit"]["R^2"].mean(), 4),
        "Lag mean (min)": round(mango_mScarlet_mean.get(construct, 0), 1),
        "Lag std (min)": round(mango_mScarlet_std.get(construct, 0), 1),
        "Lag CV (%)": round(mango_mScarlet_std.get(construct, 0) / mango_mScarlet_mean.get(construct, 1) * 100, 1),
        "Signal stability": round(stability.loc[(experiment, construct, "mScarlet")], 2),
    })

# Broccoli constructs
broccoli_constructs = {
    "F30-2xBroccoli-mScarlet": ("Broccoli", "GFP-Gext"),
    "F30-2xBroccoli-RiboJ-mScarlet": ("Broccoli", "GFP-Gext"),
    "tdBroccoli-mScarlet": ("Broccoli", "GFP-Gext"),
    "tdBroccoli-RiboJ-mScarlet": ("Broccoli", "GFP-Gext"),
}

for construct, (experiment, aptamer_read) in broccoli_constructs.items():
    clean_name = construct.replace("-mScarlet", "")

    # Aptamer row
    rows.append({
        "Experiment": experiment,
        "Construct": clean_name,
        "Reporter": "DFHBI-1T",
        "Lag detectable": "Yes",
        "SNR": round(snr.loc[(experiment, construct, aptamer_read)], 2),
        "R²": round(broccoli_aptamer_kinetics.loc[(construct, aptamer_read), "Fit"]["R^2"].mean(), 4),
        "Lag mean (min)": round(broccoli_aptamer_mean.get(construct, 0), 1),
        "Lag std (min)": round(broccoli_aptamer_std.get(construct, 0), 1),
        "Lag CV (%)": round(broccoli_aptamer_std.get(construct, 0) / broccoli_aptamer_mean.get(construct, 1) * 100, 1),
        "Signal stability": round(stability.loc[(experiment, construct, aptamer_read)], 2),
    })

    # mScarlet row
    rows.append({
        "Experiment": experiment,
        "Construct": clean_name,
        "Reporter": "mScarlet",
        "Lag detectable": "Yes",
        "SNR": round(snr.loc[(experiment, construct, "mScarlet")], 2),
        "R²": round(broccoli_mScarlet_kinetics.loc[(construct, "mScarlet"), "Fit"]["R^2"].mean(), 4),
        "Lag mean (min)": round(broccoli_mScarlet_mean.get(construct, 0), 1),
        "Lag std (min)": round(broccoli_mScarlet_std.get(construct, 0), 1),
        "Lag CV (%)": round(broccoli_mScarlet_std.get(construct, 0) / broccoli_mScarlet_mean.get(construct, 1) * 100, 1),
        "Signal stability": round(stability.loc[(experiment, construct, "mScarlet")], 2),
    })

combined_qc_table = pd.DataFrame(rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(combined_qc_table)